# WNN Experiments

This notebook implements our Weightless Neural Network.

In [3]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Load Data

In [ ]:
dataPath = Path("Data.csv")


if not dataPath.exists():
    raise FileNotFoundError(f"Data file not found at {dataPath}")

headers = pd.read_csv(dataPath, nrows=0)
cols = headers.columns.tolist()
target_col = "diseases"
symptom_cols = [col for col in cols if col != target_col]
dtype_map = {col: "uint8" for col in symptom_cols}
df = pd.read_csv(dataPath, dtype=dtype_map)

x = df[symptom_cols].to_numpy(dtype=np.float32)
y_cat = df[target_col].astype("category")

class_counts = y_cat.value_counts()
valid_classes = class_counts[class_counts >= 2].index
valid_mask = y_cat.isin(valid_classes)
x = x[valid_mask.to_numpy()]
y_cat = y_cat[valid_mask].astype("category")
y = y_cat.cat.codes.to_numpy(dtype=np.int64)
classes = y_cat.cat.categories.tolist()

TEST_SIZE = 0.15
VAL_SIZE = 0.15

X_train, X_temp, y_train, y_temp = train_test_split(x, y, test_size=TEST_SIZE + VAL_SIZE, random_state=42, stratify=y)
val_share_of_trainval = VAL_SIZE / (TEST_SIZE + VAL_SIZE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=val_share_of_trainval, random_state=42

)
print(f"Train shape: {X_train.shape}")
print(f"Val shape:   {X_val.shape}")
print(f"Test shape:  {X_test.shape}")
print(f"Num classes: {len(classes)}")




ValueError: The least populated classes in y have only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2. Classes with too few members are: [24, 34, 96, 100, 148, 160, 172, 181, 190, 191, 208, 234, 274, 301, 315, 328, 331, 343, 346, 396, 429, 441, 464, 471, 482, 489, 496, 497, 499, 501, 507, 510, 538, 597, 624, 639, 663, 671, 685, 738, 756, 763]

# Bloom Filter

In [5]:
class BloomFilter:
    def __init__(self, size: int, num_hashes: int):
        self.size = size
        self.num_hashes = num_hashes
        self.bit_array = np.zeros(size, dtype=bool)

    def _hashes(self, item: str) -> List[int]:
        return [int(hashlib.md5((item + str(i)).encode()).hexdigest(), 16) % self.size for i in range(self.num_hashes)]

    def add(self, item: str):
        for hash_val in self._hashes(item):
            self.bit_array[hash_val] = True

    def __contains__(self, item: str) -> bool:
        return all(self.bit_array[hash_val] for hash_val in self._hashes(item))

# WNN Model

In [ ]:
class BloomWNN:
    def __init__(self, input_dim: int, num_classes: int, tuple_size: int, filter_size: int, num_hashes: int, seed:int = 42) -> None:
        self.input_dim = input_dim
        self.num_classes = num_classes
        self.tuple_size = tuple_size
        rng = np.random.default_rng(seed)
        perm = rng.permutation(input_dim)
        pad = (-input_dim) % tuple_size
        if pad:
            perm = np.concatenate([perm, np.full(pad, -1, dtype=np.intp)])
        
        self.groups:np.ndarray = perm.reshape(-1, tuple_size)
        self.num_rams: int = len(self.groups)
        self.filters: List[List[BloomFilter]] = [[BloomFilter(filter_size, num_hashes) for _ in range(self.num_rams)] for _ in range(self.num_classes)]

    def tuple_value(self, x: np.ndarray, group: np.ndarray) -> int:
        val = 0
        for bitIdx in group:
            bit = int(x[bitIdx]) if bitIdx >= 0 else 0
            val = (val << 1) | bit
        return val

    def train(self, X:np.ndarray, y: np.ndarray)-> None:
        xBin = (X > 0).astype(int)
        for x, label in zip(xBin, y):
            for ram_idx, group in enumerate(self.groups):
                self.filters[int(label)][ram_idx].add(str(self.tuple_value(x, group)))
    
    def scores(self, x:np.ndarray) -> np.ndarray:
        sum = np.zeros(self.num_classes, dtype=np.float32)
        for ram_idx, group in enumerate(self.groups):
            val = str(self.tuple_value(x, group))
            for class_idx in range(self.num_classes):
                if val in self.filters[class_idx][ram_idx]:
                    sum[class_idx] += 1
        return sum / self.num_rams
    
    def predict(self, X:np.ndarray) -> np.ndarray:
        xBin = (X > 0).astype(int)
        return np.array([np.argmax(self.scores(x)) for x in xBin])

# Experiment Config

In [ ]:
@dataclass
class WNNConfig:
    name:str
    tuple_size:int
    filter_size:int
    num_hashes:int
    seed: int = 42

def trainWNNExperiment(config: WNNConfig, X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray, classes: List[str]) -> Dict[str, object]:
    model = BloomWNN(input_dim=X_train.shape[1], num_classes=len(classes), tuple_size=config.tuple_size, filter_size=config.filter_size, num_hashes=config.num_hashes, seed=config.seed)
    model.train(X_train, y_train)

    val_preds = model.predict(X_test)
    val_acc = float(accuracy_score(y_test, val_preds))
    val_prec = float(precision_score(y_test, val_preds, average='macro', zero_division=0))

    return {
        "config": config,
        "model": model,
        "val_acc": val_acc,
        "val_prec": val_prec,
    }
WNN_Experiments: List[WNNConfig] = [
    WNNConfig(name="t8  F64k  2h", tuple_size=8,  filter_size=65_536,  num_hashes=2),
    WNNConfig(name="t8  F128k 3h", tuple_size=8,  filter_size=131_072, num_hashes=3),
    WNNConfig(name="t16 F128k 3h", tuple_size=16, filter_size=131_072, num_hashes=3),
    WNNConfig(name="t4  F64k  2h", tuple_size=4,  filter_size=65_536,  num_hashes=2),
    WNNConfig(name="t32 F256k 3h", tuple_size=32, filter_size=262_144, num_hashes=3),
]

# Run Code

In [ ]:
if __name__ == "__main__":
    rows = []
    for cfg in WNN_Experiments:
        result = trainWNNExperiment(cfg, X_train, y_train, X_test, y_test, classes)
        rows.append({
            "name": cfg.name,
            "tuple_size": cfg.tuple_size,
            "filter_size": cfg.filter_size,
            "num_hashes": cfg.num_hashes,
            "val_acc": result["val_acc"],
            "val_prec": result["val_prec"],
        })
        print(f"Experiment {cfg.name}: Val Acc={result['val_acc']:.4f}, Val Prec={result['val_prec']:.4f}")
    results_df = pd.DataFrame(rows).sort_values("val_acc", ascending=False)
    results_df.plot.bar(x="name", y=["val_acc", "val_prec"], figsize=(10, 4))   